In [1]:
import importlib
import sys

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')
sys.path.insert(0, '../../../../../..')

In [2]:
# Load the data
file_path_train = '../../../../../../data/Sepsis/tensor_data/normal/sepsis_all_5_train.pkl'
sepsis_train_dataset = torch.load(file_path_train, weights_only=False)

file_path_val = '../../../../../../data/Sepsis/tensor_data/normal/sepsis_all_5_val.pkl'
sepsis_val_dataset = torch.load(file_path_val, weights_only=False)

In [3]:
# Sepsis dataset dynamic categories, features:
sepsis_all_categories = sepsis_train_dataset.all_categories

sepsis_all_categories_cat = sepsis_all_categories[0]
sepsis_all_categories_num = sepsis_all_categories[1]
for i, cat in enumerate(sepsis_all_categories_cat):
     print(f"Sepsis dynamic categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(sepsis_all_categories_num):
     print(f"Sepsis dynamic numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of numerical: {num[1]}")
print('\n')
     
# Sepsis dataset static categories, features:
sepsis_all_stat_categories = sepsis_train_dataset.all_static_categories

sepsis_all_stat_categories_cat = sepsis_all_stat_categories[0]
sepsis_all_stat_categories_num = sepsis_all_stat_categories[1]
for i, cat in enumerate(sepsis_all_stat_categories_cat):
     print(f"Sepsis static categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(sepsis_all_stat_categories_num):
     print(f"Sepsis static numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of numerical: {num[1]}")

Sepsis dynamic categorical feature: concept:name, Index position in categorical data list: 0
Sepsis amount of category labels: 18
Sepsis dynamic categorical feature: org:group, Index position in categorical data list: 1
Sepsis amount of category labels: 28
Sepsis dynamic categorical feature: lifecycle:transition, Index position in categorical data list: 2
Sepsis amount of category labels: 3


Sepsis dynamic numerical feature: case_elapsed_time, Index position in categorical data list: 0
Sepsis amount of numerical: 1
Sepsis dynamic numerical feature: event_elapsed_time, Index position in categorical data list: 1
Sepsis amount of numerical: 1
Sepsis dynamic numerical feature: day_in_week, Index position in categorical data list: 2
Sepsis amount of numerical: 1
Sepsis dynamic numerical feature: seconds_in_day, Index position in categorical data list: 3
Sepsis amount of numerical: 1
Sepsis dynamic numerical feature: Leucocytes, Index position in categorical data list: 4
Sepsis amount of nu

In [4]:
# Create lists with name of Encoder features (input) and decoder features (input & output)

# Encoder features (fixed): use only requested features
enc_feat_cat = ['concept:name', 'org:group']
enc_feat_num = ['case_elapsed_time']
enc_feat = [enc_feat_cat, enc_feat_num]
print("Input features encoder: ", enc_feat)

# Decoder features (unused by C-LSTM training cell, kept for consistency)
dec_feat_cat = ['concept:name']
dec_feat_num = []
dec_feat = [dec_feat_cat, dec_feat_num]
print("Features decoder: ", dec_feat)

Input features encoder:  [['concept:name', 'org:group'], ['case_elapsed_time']]
Features decoder:  [['concept:name'], []]


In [5]:
import suffix_pred.models.C_LSTM
importlib.reload(suffix_pred.models.C_LSTM)
from suffix_pred.models.C_LSTM import FullShared_Join_LSTM

# Size hidden layer
hidden_size = 50

# Number of LSTM cells
num_layers = 1

# STANDARD: One numerical output to predict
input_size = 1

# C-LSTM uses dynamic prefix features only
model_feat = enc_feat

# Output size: activity classes only
activity_feature_name = 'concept:name'
activity_feature_index = next(i for i, cat in enumerate(sepsis_all_categories_cat) if cat[0] == activity_feature_name)
output_size_act = sepsis_all_categories_cat[activity_feature_index][1]

# C-LSTM model initialization
model = FullShared_Join_LSTM(data_set_categories=sepsis_all_categories,
                             hidden_size=hidden_size,
                             num_layers=num_layers,
                             model_feat=model_feat,
                             input_size=input_size,
                             output_size_act=output_size_act)

Data set categories:  ([('concept:name', 18, {'Admission IC': 1, 'Admission NC': 2, 'CRP': 3, 'EOS': 4, 'ER Registration': 5, 'ER Sepsis Triage': 6, 'ER Triage': 7, 'IV Antibiotics': 8, 'IV Liquid': 9, 'LacticAcid': 10, 'Leucocytes': 11, 'Release A': 12, 'Release B': 13, 'Release C': 14, 'Release D': 15, 'Release E': 16, 'Return ER': 17}), ('org:group', 28, {'?': 1, 'A': 2, 'B': 3, 'C': 4, 'D': 5, 'E': 6, 'EOS': 7, 'F': 8, 'G': 9, 'H': 10, 'I': 11, 'J': 12, 'K': 13, 'L': 14, 'M': 15, 'N': 16, 'O': 17, 'P': 18, 'Q': 19, 'R': 20, 'S': 21, 'T': 22, 'U': 23, 'V': 24, 'W': 25, 'X': 26, 'Y': 27}), ('lifecycle:transition', 3, {'EOS': 1, 'complete': 2})], [('case_elapsed_time', 1, {}), ('event_elapsed_time', 1, {}), ('day_in_week', 1, {}), ('seconds_in_day', 1, {}), ('Leucocytes', 1, {}), ('CRP', 1, {}), ('LacticAcid', 1, {})])
Model input features:  [['concept:name', 'org:group'], ['case_elapsed_time']]


Embeddings:  ModuleList(
  (0): Embedding(18, 8)
  (1): Embedding(28, 10)
)
Total embedd

/home/PSPLab/.local/share/virtualenvs/decision_aware_augmentation_for_PPM-0DzgvVpC/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [6]:
import suffix_pred.loss
importlib.reload(suffix_pred.loss)
from suffix_pred.loss import Loss

loss_obj = Loss()

In [7]:
import suffix_pred.train
importlib.reload(suffix_pred.train)
from suffix_pred.train import CTraining

from torch.optim.lr_scheduler import ReduceLROnPlateau

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Start learning rate
learning_rate = 0.001
weight_decay = 0.0

# Optimizer and Scheduler
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=15, min_lr=1e-8)


# Keep consistent across all models
# Epochs / Batch size
num_epochs = 100
batch_size = 128

# Shuffle data
shuffle = True

optimize_values = {"optimizer": optimizer,
                   "scheduler": scheduler,
                   "epochs": num_epochs,
                   "mini_batches": batch_size,
                   "shuffle": shuffle}

# Activity feature index and EOS id for activity-only next-event prediction
activity_feature_name = 'concept:name'
concept_name_id = next(i for i, cat in enumerate(sepsis_all_categories_cat) if cat[0] == activity_feature_name)
activity_label_to_id = sepsis_all_categories_cat[concept_name_id][2]
eos_id = activity_label_to_id.get('EOS')

model_output_path = "../../../../../../models/Sepsis/clean/Sepsis_C_LSTM_v1_clean.pkl"
os.makedirs(os.path.dirname(model_output_path), exist_ok=True)

trainer = CTraining(device=device,
                    model=model,
                    data_train=sepsis_train_dataset,
                    data_val=sepsis_val_dataset,
                    optimize_values=optimize_values,
                    concept_name_id=concept_name_id,
                    eos_id=eos_id,
                    loss_obj=loss_obj,
                    save_model_n_th_epoch=1,
                    saving_path=model_output_path)

# Train the model (activity sequence modeling via next-activity training)
trainer.train()

Device: cuda
Device:  cuda
Optimizer:  Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0
)
Scheduler:  <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x7fa1caa04e00>
Epochs:  100
Mini baches:  128
Shuffle batched dataset:  True


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [1/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 1.9360
Validation: Avg Validation Loss: 1.5185
Validation Loss for Scheduler: 1.5185
saving model
Epoch [2/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 1.3845
Validation: Avg Validation Loss: 1.2632
Validation Loss for Scheduler: 1.2632
saving model
Epoch [3/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 1.2114
Validation: Avg Validation Loss: 1.1497
Validation Loss for Scheduler: 1.1497
saving model
Epoch [4/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 1.1293
Validation: Avg Validation Loss: 1.0798
Validation Loss for Scheduler: 1.0798
saving model
Epoch [5/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 1.0629
Validation: Avg Validation Loss: 1.0398
Validation Loss for Scheduler: 1.0398
saving model
Epoch [6/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 1.0319
Validation: Avg Validation Loss: 1.0292
Validat

([1.936020482670177,
  1.3845213883882994,
  1.2113509154939033,
  1.1292823700161723,
  1.0629121289624797,
  1.0319169287557726,
  1.0037533029333336,
  0.9815085963769392,
  0.9649961544321729,
  0.9614088024411883,
  0.9387012549809047,
  0.934567394194665,
  0.923622329513748,
  0.9187593552973363,
  0.908166876086941,
  0.908099306094182,
  0.8956113527347516,
  0.8902313167398627,
  0.8785599880404287,
  0.8712385412934539,
  0.8727089956209257,
  0.8583504921430117,
  0.8602041501503486,
  0.8509446351559131,
  0.8449344991089461,
  0.8400458167125653,
  0.8320112878626044,
  0.82322825936528,
  0.8221936009146951,
  0.8116340544316676,
  0.7995759310660424,
  0.8002633932348969,
  0.7907929397248602,
  0.7868929497607342,
  0.7797350875743023,
  0.7742313053700831,
  0.7657945953406297,
  0.7370475522883526,
  0.730569724138681,
  0.7220353460931158,
  0.7225796595796362,
  0.7243729109887953,
  0.7218251406372368,
  0.7193112706209158,
  0.7172809336092565,
  0.71375938785540